# The reserved period: checks, then one scoring

[Open in Colab](https://colab.research.google.com/github/Gaire-commits/entropy-gate-pipeline/blob/main/notebooks/reserved_evaluation.ipynb)

Two frozen protocols share the period 2024-01-03 to 2026-10-12: `protocols/volatility_v2.yaml` and `protocols/portfolio_v2.yaml`. **It is reserved for these specifications but previously exposed through related research**; `protocols/reserved_period_disclosure.md` lists every prior use.

This notebook runs in steps. Set `STEP` below, then Runtime → Run all. Each step includes the ones before it, and finished files are kept. Results print here; nothing is downloaded. Paste the output into the chat.

| STEP | What it does | Needs | Time |
|---|---|---|---:|
| **1** | The data checks: SPY and BIL against the recorded returns, official-open coverage, whether missing opens cluster on extreme days, quoted spreads. **Scores nothing.** For the reserved period it prints coverage counts only. | Colab Secrets | about 20 minutes |
| **2** | The baseline gate on the official open, from daily bars. **No model uses PE, and nothing after 2024-01-02 is read.** | nothing more | about 10 minutes more |
| **3** | **Scores the volatility protocol on the reserved period, once,** from daily bars. The portfolio protocol was scored on 2026-11-17. It cannot be undone. | `CONFIRM` below, typed exactly | about 15 minutes more |

**Don't save this notebook to GitHub from an old copy.** The code it runs is hash-checked by `tests/test_frozen.py`, so a change to a protocol or its code shows up.

In [ ]:
STEP = 0           # 0 skips the large-cap checks; 1 or 2 run them. STEP 3 has run: both protocols are scored
CONFIRM = ''       # for STEP 3 only: 'score the reserved period'
EXPLORE = ''       # after STEP 1: 'har_x', 'signals', 'pe_long', or '' for none
MIDCAP = False     # the HAR-X replication on mid caps: ran 2026-11-18, REPLICATES
FORWARD = False    # the HAR-X forward test: set True every few months; fetches both universes to yesterday (about 30 minutes)

In [ ]:
import os, sys
IN_COLAB = 'google.colab' in sys.modules
REPO = '/content/entropy-gate-pipeline' if IN_COLAB else os.environ.get('PORTFOLIO_REPO', os.getcwd())
if IN_COLAB:
    if os.path.isdir(f'{REPO}/.git'):
        !cd {REPO} && git pull -q --ff-only
    else:
        !git clone -q https://github.com/Gaire-commits/entropy-gate-pipeline.git {REPO}
    %cd {REPO}
    !git log --oneline -1
    from google.colab import userdata
    try:
        with open('.env', 'w') as f:
            f.write(f"APCA_API_KEY_ID={userdata.get('APCA_API_KEY_ID')}\nAPCA_API_SECRET_KEY={userdata.get('APCA_API_SECRET_KEY')}\n")
        print('keys written to a local .env (never printed, never committed)')
    except Exception as exc:
        print('No Alpaca keys in Colab Secrets:', type(exc).__name__, '- add them from the key icon, then run this cell again.')
else:
    os.chdir(REPO)
for m in [m for m in sys.modules if m == 'src' or m.startswith('src.')]:
    del sys.modules[m]
sys.path[:0] = [REPO, f'{REPO}/scripts']

from pathlib import Path
REF = os.environ.get('RESERVED_REF', '/content/reserved/reference')
WORK = os.environ.get('RESERVED_WORK', '/content/reserved')
DRIVE = '/content/drive/MyDrive/entropy-gate-pipeline-data/data'
Path(REF).mkdir(parents=True, exist_ok=True)

## Step 1: the data checks

In [ ]:
if STEP >= 1:
    !python scripts/fetch_reference.py --universe universe/sp500.csv --start 2016-01-04 --end 2026-10-13 --out {REF} --parts daily,corporate_actions,meta,auctions 2>&1 | tail -4
    !python scripts/official_prices.py --reference {REF}
    !python scripts/protocol_checks.py --reference {REF}
else:
    print('STEP is 0 - the large-cap checks were skipped')

## Step 2: the baseline gate on the official open

In [ ]:
if STEP >= 2:
    # daily bars only (protocols/amendment_4.md): no Drive, no 5-minute cache
    !python scripts/vol_forecast.py --stage baseline --reference {REF} --official {REF}/official_prices.parquet 2>&1 | grep -v Warning
else:
    print('STEP is', STEP, '- skipped')

## Step 3: locked

This cell scores the reserved period, once. It does nothing unless `STEP = 3` and `CONFIRM` is typed exactly, and it is not to run until Kshitiz decides. Record the outcome in `RESULTS.md` whatever it is.

In [ ]:
if STEP >= 3:
    assert CONFIRM == 'score the reserved period', "set CONFIRM exactly to run this step"
    # The portfolio protocol was scored on 2026-11-17 and is not run again. Volatility on daily bars only (protocols/amendment_4.md).
    print('=' * 30, 'VOLATILITY')
    !python scripts/vol_forecast.py --stage reserved --score-reserved --reference {REF} --official {REF}/official_prices.parquet 2>&1 | grep -v Warning
else:
    print('STEP is', STEP, '- the reserved period was not scored')

## Exploration (not registered tests)

Both protocols are scored, so the data from 2016 to 2026 is open for exploring. Anything found is a lead: it is written as a rule and tested on trading days after 2026-11-17. Every test run is counted. Daily bars only, no Drive.

- `'signals'`: ten simple daily features against the next day's return, the next day's open-to-close return, the next two days' return, and next-day variance beyond what volatility history predicts. About 40 tests, with a false-discovery correction.
- `'pe_long'`: PE once the long-run volatility level is in the model (run 2026-11-17; recorded in RESULTS.md).
- `'har_x'`: HAR plus the long-run level, then adding the signal search's features in blocks (the range against its usual level, recent returns, the largest return, volume shock), each measured out of sample, 2017 to 2026.

In [ ]:
if EXPLORE == 'har_x':
    !python scripts/explore_har_x.py --reference {REF} --official {REF}/official_prices.parquet 2>&1 | grep -v Warning
elif EXPLORE == 'signals':
    !python scripts/explore_signals.py --reference {REF} --official {REF}/official_prices.parquet 2>&1 | grep -v Warning
elif EXPLORE in ('pe_long', True):
    !python scripts/explore_pe_long.py --reference {REF} --official {REF}/official_prices.parquet 2>&1 | grep -v Warning
else:
    print('EXPLORE is off')

## Replication: the volatility model on mid caps (frozen 2026-11-18)

The model found on large caps (HAR plus the long-run level, recent returns and volume) is frozen in `protocols/har_x_replication_v1.yaml` and run once on the S&P MidCap 400, whose daily volatility has not been used for anything. It downloads the mid caps' daily bars and auction prints, takes the official open, and judges the model by the protocol's rule. Daily bars only, no Drive. Paste the output into the chat.

In [ ]:
if MIDCAP:
    MID = '/content/midcap/reference'
    !python scripts/fetch_reference.py --universe universe/sp400.csv --start 2016-01-04 --end 2026-10-13 --out {MID} --parts daily,corporate_actions,meta,auctions 2>&1 | tail -4
    !python scripts/official_prices.py --reference {MID} 2>&1 | grep -A14 "coverage by year"
    !python scripts/har_x_replication.py --reference {MID} --official {MID}/official_prices.parquet 2>&1 | grep -v Warning
else:
    print('MIDCAP is off')

## Forward test: the volatility model on new trading days (registered 2026-11-18)

`protocols/har_x_forward_v1.yaml`. The frozen model against HAR plus the long-run level on the S&P 500 and S&P MidCap 400, scoring only target dates from 2026-11-19. It downloads both universes' daily bars and auction prints up to yesterday's close, every time. **The verdict comes once, at 500 target dates (about November 2028).** Before that each run is a progress report, labelled interim. Run it every few months and paste the output.

The first useful run is after about 125 trading days (around May 2027); a run before then scores few or no days.

In [ ]:
if FORWARD:
    import datetime
    from pathlib import Path
    # Up to yesterday: the Alpaca plan refuses SIP data from the last 15 minutes, and only completed sessions count.
    END = (datetime.date.today() - datetime.timedelta(days=1)).isoformat()
    ok = True
    for NAME, LIST in (('sp500', 'universe/sp500.csv'), ('sp400', 'universe/sp400.csv')):
        OUT = f'/content/forward/{NAME}'
        !python scripts/fetch_reference.py --universe {LIST} --start 2016-01-04 --end {END} --out {OUT} --parts daily,corporate_actions,meta,auctions --force 2>&1 | tail -2
        if not Path(f'{OUT}/daily/daily_sip_raw.parquet').exists() or not Path(f'{OUT}/auctions.parquet').exists():
            print(f'STOPPED: the {NAME} download did not finish (see the lines above); nothing was scored.')
            ok = False
            break
        !python scripts/official_prices.py --reference {OUT} 2>&1 | tail -1
    if ok:
        !python scripts/har_x_forward.py --sp500 /content/forward/sp500 --sp400 /content/forward/sp400 2>&1 | grep -v Warning
else:
    print('FORWARD is off')